# Getting Started

> An introduction to using the experimental features of Ragas

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"

In [21]:
from ragas_experimental.utils import get_test_directory

In [22]:
from ragas_experimental import Project

In [40]:
root_dir = "."

In [41]:
p = Project(
    project_id="test",
    backend="local",
    root_dir=root_dir,
)

p

In [42]:
from ragas_experimental import BaseModel
import typing as t

class TestDataRow(BaseModel):
    id: t.Optional[int]
    query: str
    expected_output: str

In [43]:
dataset = p.create_dataset(
    name="test_dataset",
    model=TestDataRow,
)

dataset

Dataset(name='test_dataset', model=TestDataRow, len=0)

In [44]:
import pandas as pd
df = pd.read_csv("/Users/shahules/Downloads/test_data.csv")

In [45]:
for i, row in df.iterrows():
    row = TestDataRow(id=i, query=row["query"], expected_output=row['expected_output'])
    dataset.append(row)

dataset

Dataset(name='test_dataset', model=TestDataRow, len=30)

In [46]:
from openai import AsyncOpenAI
client = AsyncOpenAI()

async def my_app_endpoint(query: str) -> str:
    response = await client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": query},
        ],
    )
    return response.choices[0].message.content



In [47]:
from ragas_experimental.metric import DiscreteMetric
from ragas_experimental.llm import ragas_llm

llm = ragas_llm(provider="openai",model="gpt-4o",client=client)

metric = DiscreteMetric(
    name="accuracy",
    llm=llm,
    prompt="Given query {query}, response from pipeline is {response}. Does the response match the expected output {expected_output}?",
    values=["pass", "fail"]
    )


In [48]:
from ragas_experimental.metric import MetricResult

class ExperimentDataRow(TestDataRow):
    response: str 
    accuracy: MetricResult


@p.experiment(ExperimentDataRow)
async def run_experiment(row: TestDataRow):
    response = await my_app_endpoint(row.query)
    score = await metric.ascore(
        query=row.query,
        response=response,
        expected_output=row.expected_output
    )

    experiment_view = ExperimentDataRow(
        id=row.id,
        query=row.query,
        expected_output=row.expected_output,
        response=response,
        accuracy=score,
    )
    return experiment_view

In [49]:
await run_experiment.run_async(dataset)

Running experiment: 100%|██████████| 60/60 [00:04<00:00, 12.55it/s]


Experiment(name=xenodochial_wozniak, model=ExperimentDataRow, len=30)